In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

In [2]:
df = pd.read_csv('../Car-details.csv')
df.sample(5)

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
5694,Ford Fiesta 1.4 Duratec ZXI,2008,136000,200185,Petrol,Individual,Manual,First Owner,16.6 kmpl,1388 CC,68 bhp,"16.3@ 2,000(kgm@ rpm)",5.0
73,Hyundai i10 Magna,2011,235000,60000,Petrol,Individual,Manual,Second Owner,20.36 kmpl,1197 CC,78.9 bhp,111.7Nm@ 4000rpm,5.0
2355,Maruti Swift ZXI Plus,2018,720000,20000,Petrol,Individual,Manual,First Owner,22.0 kmpl,1197 CC,81.80 bhp,113Nm@ 4200rpm,5.0
1642,Maruti Swift Dzire VXI,2016,500000,30000,Petrol,Individual,Manual,First Owner,20.85 kmpl,1197 CC,83.14 bhp,115Nm@ 4000rpm,5.0
6284,Maruti Ertiga SHVS VDI,2019,925000,50000,Diesel,Dealer,Manual,First Owner,24.52 kmpl,1248 CC,88.5 bhp,200Nm@ 1750rpm,7.0


In [3]:
df['mileage'] = df['mileage'].str.extract('(\d+\.?\d*)').astype(float)
df['engine'] = df['engine'].str.extract('(\d+\.?\d*)').astype(float)
df['max_power'] = df['max_power'].str.extract('(\d+\.?\d*)').astype(float)
df['torque'] = df['torque'].str.extract('(\d+\.?\d*)').astype(float)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8128 entries, 0 to 8127
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   name           8128 non-null   object 
 1   year           8128 non-null   int64  
 2   selling_price  8128 non-null   int64  
 3   km_driven      8128 non-null   int64  
 4   fuel           8128 non-null   object 
 5   seller_type    8128 non-null   object 
 6   transmission   8128 non-null   object 
 7   owner          8128 non-null   object 
 8   mileage        7907 non-null   object 
 9   engine         7907 non-null   object 
 10  max_power      7913 non-null   object 
 11  torque         7906 non-null   object 
 12  seats          7907 non-null   float64
dtypes: float64(1), int64(3), object(9)
memory usage: 825.6+ KB


In [6]:
df.isna().sum()

name               0
year               0
selling_price      0
km_driven          0
fuel               0
seller_type        0
transmission       0
owner              0
mileage          221
engine           221
max_power        215
torque           222
seats            221
dtype: int64

In [7]:
df.shape


(8128, 13)

In [12]:
for col in df.select_dtypes(include='O').columns:
    print(f'Column: {col}')
    print(f'Cardinality: {df[col].nunique()}')
    print(df[col].unique())
    print(df[col].value_counts(normalize=True))
    print()


Column: name
Cardinality: 2058
['Maruti Swift Dzire VDI' 'Skoda Rapid 1.5 TDI Ambition'
 'Honda City 2017-2020 EXi' ... 'Tata Nexon 1.5 Revotorq XT'
 'Ford Freestyle Titanium Plus Diesel BSIV'
 'Toyota Innova 2.5 GX (Diesel) 8 Seater BS IV']
name
Maruti Swift Dzire VDI              0.015871
Maruti Alto 800 LXI                 0.010089
Maruti Alto LXi                     0.008735
BMW X4 M Sport X xDrive20d          0.007628
Maruti Swift VDI                    0.007505
                                      ...   
Maruti 800 DX BSII                  0.000123
Ford Figo Aspire Titanium Diesel    0.000123
Hyundai Verna CRDi 1.6 SX           0.000123
Maruti Baleno Alpha Diesel          0.000123
Tata New Safari Dicor VX 4X2        0.000123
Name: proportion, Length: 2058, dtype: float64

Column: fuel
Cardinality: 4
['Diesel' 'Petrol' 'LPG' 'CNG']
fuel
Diesel    0.541585
Petrol    0.446727
CNG       0.007013
LPG       0.004675
Name: proportion, dtype: float64

Column: seller_type
Cardinality: 3


In [13]:
df.duplicated().sum()

np.int64(1202)

In [15]:
df =df.drop_duplicates()
df.head()

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.4 kmpl,1248 CC,74 bhp,190Nm@ 2000rpm,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14 kmpl,1498 CC,103.52 bhp,250Nm@ 1500-2500rpm,5.0
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.7 kmpl,1497 CC,78 bhp,"12.7@ 2,700(kgm@ rpm)",5.0
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.0 kmpl,1396 CC,90 bhp,22.4 kgm at 1750-2750rpm,5.0
4,Maruti Swift VXI BSIII,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.1 kmpl,1298 CC,88.2 bhp,"11.5@ 4,500(kgm@ rpm)",5.0


In [16]:
df.duplicated().sum()

np.int64(0)

In [5]:
X = df.drop(columns='selling_price')
Y = df.selling_price.copy()
print(X.shape, Y.shape)

(8128, 12) (8128,)


In [6]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
print(X_train.shape, Y_train.shape)
print(X_test.shape, Y_test.shape)

(6502, 12) (6502,)
(1626, 12) (1626,)


In [18]:
num_cols = X_train.select_dtypes(include='number').columns.tolist()
cat_cols = [col for col in X_train.columns if col not in num_cols]
print(num_cols)
print(cat_cols)

['year', 'km_driven', 'mileage', 'engine', 'max_power', 'torque', 'seats']
['name', 'fuel', 'seller_type', 'transmission', 'owner']


In [12]:
num_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy = 'median')),
    ('scaler', StandardScaler())
])

cat_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)

])

regressor = RandomForestRegressor(
    n_estimators=10, max_depth=5, random_state=42
)

rf_model = Pipeline(steps=[
    ('pre', preprocessor),
    ('reg', regressor)
])
rf_model.fit(X_train, Y_train)

,steps,"[('pre', ...), ('reg', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [17]:
Y_train_pred = rf_model.predict(X_train)
train_rmse = root_mean_squared_error(Y_train, Y_train_pred)
print(f'Train RMSE: {train_rmse:,.3f}')
Y_test_pred = rf_model.predict(X_test)
test_rmse = root_mean_squared_error(Y_train, Y_train_pred)
print(f'Test RMSE: {test_rmse:,.3f}')

Train RMSE: 175,239.635
Test RMSE: 175,239.635
